### Loading all files

In [1]:
# loads the files from data_files folder 

import json
import os
from langchain_community.document_loaders import PyPDFLoader, TextLoader, UnstructuredWordDocumentLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from pathlib import Path

#load all the files from data_files folder

def _sanitize_metadata_value(value):
    if isinstance(value, (str, int, float, bool)):
        return value
    if value is None:
        return None
    if isinstance(value, Path):
        return str(value)
    if hasattr(value, 'isoformat'):
        try:
            return value.isoformat()
        except Exception:
            pass
    if isinstance(value, (list, tuple, set)):
        return ', '.join(str(item) for item in value)
    if isinstance(value, dict):
        return json.dumps(value, default=str, ensure_ascii=True)
    return str(value)

def sanitize_metadata(metadata):
    cleaned = {}
    for key, value in metadata.items():
        sanitized_value = _sanitize_metadata_value(value)
        if sanitized_value is not None:
            cleaned[str(key)] = sanitized_value
    return cleaned

def load_files(data_folder):
    """
    Loads all files(.pdf, .txt, .docx) from the specified data folder and returns a list of Document objects.
    """
    data_folder = Path(data_folder)
    all_docs = []
    loaded_files_count = 0

    # Define loaders for each file type
    loaders = {
        '.pdf': PyPDFLoader,
        '.txt': lambda path: TextLoader(path, encoding='utf-8', autodetect_encoding=True),
        '.docx': UnstructuredWordDocumentLoader,
    }

    for file_path in data_folder.iterdir():
        if file_path.is_file():
            suffix = file_path.suffix.lower()
            if suffix in loaders:
                try:
                    loader = loaders[suffix](str(file_path))
                    docs = loader.load()
                    # Add metadata to documents
                    for i, doc in enumerate(docs):
                        extra_metadata = {
                            "source": str(file_path),
                            "file_name": file_path.name,
                            "file_type": suffix[1:],  # Remove the dot
                        }
                        if suffix == '.pdf':
                            extra_metadata["page_number"] = i + 1
                        doc.metadata.update(extra_metadata)
                        doc.metadata = sanitize_metadata(doc.metadata)
                    all_docs.extend(docs)
                    loaded_files_count += 1
                except Exception as e:
                    print(f"Error loading {file_path}: {e}")
            else:
                print(f"Unsupported file format: {file_path.name}")
    
    print(f"Loaded {loaded_files_count} files.")
    return all_docs


c:\Users\akash\OneDrive\Desktop\RAG_chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#test if load_files function works
if __name__ == "__main__":
    data_folder = "..//data_files"
    documents = load_files(data_folder)
    print(f"Total documents loaded: {len(documents)}")
    for doc in documents[:5]:  # Print metadata of the first 5 documents
        print(doc.metadata)

Loaded 8 files.
Total documents loaded: 30
{'source': '..\\data_files\\05-03-ai-powered-supply-chain-startup-pando-lands-30m-investment.txt', 'file_name': '05-03-ai-powered-supply-chain-startup-pando-lands-30m-investment.txt', 'file_type': 'txt'}
{'source': '..\\data_files\\05-03-ai-replace-tv-writers-strike.txt', 'file_name': '05-03-ai-replace-tv-writers-strike.txt', 'file_type': 'txt'}
{'source': '..\\data_files\\05-03-databricks-acquires-ai-centric-data-governance-platform-okera.txt', 'file_name': '05-03-databricks-acquires-ai-centric-data-governance-platform-okera.txt', 'file_type': 'txt'}
{'producer': 'WeasyPrint 54.3', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data_files\\Akash_Raj_23117014_Resume_26.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_name': 'Akash_Raj_23117014_Resume_26.pdf', 'file_type': 'pdf', 'page_number': 1}
{'producer': 'Skia/PDF m126', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/12

In [3]:
print(len(documents))
print(type(documents[0]))

30
<class 'langchain_core.documents.base.Document'>


### Splitting documents in chunks

In [4]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    '''Splits documents into chunks'''
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    split_docs = text_splitter.split_documents(documents)  #returns a list of documents
    print(f"Total chunks created: {len(split_docs)}")
    return split_docs

In [5]:
chunks = split_documents(documents)
print(chunks[0].metadata)
print(f"Type of chunks[0]: {type(chunks[0])}")


Total chunks created: 76
{'source': '..\\data_files\\05-03-ai-powered-supply-chain-startup-pando-lands-30m-investment.txt', 'file_name': '05-03-ai-powered-supply-chain-startup-pando-lands-30m-investment.txt', 'file_type': 'txt'}
Type of chunks[0]: <class 'langchain_core.documents.base.Document'>


### Embeddings and Vector Store

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple, Union
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """
    Manages embedding generation.
    """
    def __init__(self, embedding_model_name: str = 'all-MiniLM-L6-v2'):
        self.embedding_model = embedding_model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Loads the embedding model."""
        try:
            self.model = SentenceTransformer(self.embedding_model)
            print(f"Loaded embedding model: {self.embedding_model}")
        except Exception as e:
            print(f"Error loading embedding model: {e}")
            raise


    def generate_embeddings(self, texts: List[Union[str, Document]])->np.ndarray:
        """Generates embeddings for a list of texts or Document objects.
        Args:
            texts (List[Union[str, Document]]): List of strings or Document objects to embed.
            Returns: np.ndarray: Array of embeddings.
        """
        # try:
        #     text_list = []
        #     metadata_list = []

        #     for doc in texts:
        #         if isinstance(doc, Document):
        #             text_list.append(doc.page_content)
        #             metadata_list.append(doc.metadata)
        #         elif isinstance(doc, str):
        #             text_list.append(doc)
        #             metadata_list.append({})
        #         else:
        #             raise ValueError(f"Unsupported type: {type(doc)}. Expected str or Document.")
        #     embeddings = self.model.encode(
        #         text_list,
        #         show_progress_bar=True,
        #         batch_size=32,
        #         normalize_embeddings=True,
        #     )
        #     return embeddings, metadata_list, text_list
        # except Exception as e:
        #     print(f"Error generating embeddings: {e}")
        #     raise

        try:
            # Extract text content
            # metadata = self._sanitize_metadata(dict(doc.metadata) if doc.metadata else {})
            text_contents = [doc.page_content if isinstance(doc, Document) else doc for doc in texts]
            embeddings = self.model.encode(text_contents, show_progress_bar=True, batch_size=32, normalize_embeddings=True)
            return np.array(embeddings)
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise
    

In [8]:
embedding_manager = EmbeddingManager()
embeddings = embedding_manager.generate_embeddings(chunks)

print("First 5 metadata, text, and their embedding vectors:")
for i in range(5):
    # print(f"Metadata: {metadata_list[i]}")
    # print(f"Text: {text_list[i]}")
    print(f"Embedding vector (first 5 values): {embeddings[i][:5]}")
    print("-" * 50)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7525.32it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded embedding model: all-MiniLM-L6-v2


Batches: 100%|██████████| 3/3 [00:02<00:00,  1.41it/s]

First 5 metadata, text, and their embedding vectors:
Embedding vector (first 5 values): [ 0.02821259 -0.06104733  0.0241313  -0.04618432  0.0522055 ]
--------------------------------------------------
Embedding vector (first 5 values): [-0.03588095 -0.03687634  0.02861731 -0.06605814  0.06482797]
--------------------------------------------------
Embedding vector (first 5 values): [-0.07824741 -0.01432543  0.00976842 -0.02931419  0.09825265]
--------------------------------------------------
Embedding vector (first 5 values): [-0.00242981 -0.00637816  0.02274021 -0.08020833  0.07078611]
--------------------------------------------------
Embedding vector (first 5 values): [-0.02059556 -0.0593768   0.03396064 -0.03689031  0.08263365]
--------------------------------------------------


In [9]:
class VectorStore:
    """
    Manages storage and retrieval of embeddings. 
    """

    def __init__(self, collection_name: str = "documents_chunks", persist_directory: str = "../vector_database"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_chromadb()

    def _initialize_chromadb(self):
        """Initializes the ChromaDB client and collection."""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path = self.persist_directory)
            self.collection = self.client.create_collection(
                name=self.collection_name, 
                get_or_create=True,
                metadata={"description": "Collection of document chunks and their embeddings"}
            )
            print(f"Collection '{self.collection_name}' is ready.")
        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise
    
    def _sanitize_metadata(self, metadata: dict) -> dict:
        """Ensures all metadata values are ChromaDB-compatible types."""
        sanitized = {}
        for k, v in metadata.items():
            if isinstance(v, (str, int, float, bool)):
                sanitized[k] = v
            elif v is None:
                sanitized[k] = ""          # Replace None with empty string
            else:
                sanitized[k] = str(v)      # Coerce lists, dicts, etc. to string
        return sanitized

    def add_docs(self, docs: List[Union[str, Document]], embeddings: np.ndarray):
        """ 
        Adds documents and their embeddings to the collection.
        Args:
            docs (List[Union[str, Document]]): List of documents or strings to add.
            embeddings (np.ndarray): Corresponding array of embeddings.
        """
        print(f"Adding {len(docs)} documents to the vector store...")

        #prepare data for insertion
        ids = []
        metadatas = []
        document_texts = []
        embedding_lists = []

        for i, (doc, embedding) in enumerate(zip(docs, embeddings)):
            doc_id = str(uuid.uuid4()) # Generate a unique ID for each document chunk
            ids.append(doc_id)

            #prepare metadata for each chunk
            metadata = self._sanitize_metadata(dict(doc.metadata)) if doc.metadata else {}
            metadata['doc_index']=i
            metadata['content_len']=len(doc.page_content)
            metadatas.append(metadata)

            document_texts.append(doc.page_content if isinstance(doc, Document) else str(doc))
            embedding_lists.append(embedding.tolist())

        # Insert data into the collection
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=document_texts,
                embeddings=embedding_lists
            )
            print(f"Successfully added {len(docs)} documents to the vector store.")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


In [10]:
vector_store = VectorStore()
vector_store.add_docs(chunks, embeddings)

Collection 'documents_chunks' is ready.
Adding 76 documents to the vector store...
Successfully added 76 documents to the vector store.


### Retrieval Layer

In [11]:
class RAGretriever:
    """
    Handles querying the vector store and retrieving relevant document chunks based on similarity to the query. 
    """
    def __init__(self, vector_store_manager: VectorStore, embedding_manager: EmbeddingManager, threshold_dist: float = 0.7):
        self.vector_store_manager = vector_store_manager
        self.embedding_manager = embedding_manager
        self.threshold_dist = threshold_dist

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict[str, Any]]:
        """Retrieves relevant document chunks based on similarity to the query.
        Args:
            query (str): The input query for which to retrieve relevant documents.
            top_k (int): The number of top similar documents to retrieve.
        Returns:
            List[Dict[str, Any]]: A list of retrieved documents with their metadata and similarity scores.
        """
        # Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Retrieve all documents and their embeddings from the vector store
        try:
            results = self.vector_store_manager.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
                include=["documents", "metadatas", "distances"]
            )

            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:  # Check if there are any retrieved documents
                for rank, (doc, metadata, distance, id) in enumerate(zip(results['documents'][0], results['metadatas'][0], results['distances'][0], results['ids'][0])):
                    # similarity_score = 1 - distance  # Convert distance to similarity score (assuming distance is between 0 and 1)
                    if distance> self.threshold_dist:
                        break  # Stop if distance exceeds the threshold

                    retrieved_docs.append({
                        "document": doc,
                        "metadata": metadata,
                        # "similarity_score": similarity_score,
                        "distance": distance,
                        "doc_id": id,
                        "rank": rank + 1  # Rank starts at 1
                    })
            else:
                print("No documents retrieved for the query.")
                return []
            return retrieved_docs
        except Exception as e:
            print(f"Error retrieving documents: {e}")
            raise

In [12]:
rag_retriever = RAGretriever(vector_store, embedding_manager, threshold_dist=3.0)

In [13]:
query = "What are Akash Raj's skills and experience?"

retrieved_docs = rag_retriever.retrieve(query, top_k=10) 

print("Retrieved documents:")
for doc in retrieved_docs:
    print(f"Rank: {doc['rank']}, Distance: {doc['distance']:.4f}")
    print(f"Document: {doc['document'][:200]}...")  # First 200 chars
    print(f"Metadata: {doc['metadata']}")
    print("-" * 50) 

Batches: 100%|██████████| 1/1 [00:00<00:00, 42.77it/s]

Retrieved documents:
Rank: 1, Distance: 1.3987
Document: 52+ Amenities across 3 levels

1) LEVEL ONE RETREAT 

Badminton court, Sr. citizen plaza, Banquet Hall, Squash Court & Kids play area

2) E-DECK RETREAT

Swimming pool, Glass house cafe, Yoga Deck, Cr...
Metadata: {'file_name': 'Seabreez Godrej Project Details.docx', 'content_len': 873, 'source': '..\\data_files\\Seabreez Godrej Project Details.docx', 'doc_index': 73, 'file_type': 'docx', 'page_number': ''}
--------------------------------------------------
Rank: 2, Distance: 1.3987
Document: 52+ Amenities across 3 levels

1) LEVEL ONE RETREAT 

Badminton court, Sr. citizen plaza, Banquet Hall, Squash Court & Kids play area

2) E-DECK RETREAT

Swimming pool, Glass house cafe, Yoga Deck, Cr...
Metadata: {'content_len': 873, 'doc_index': 73, 'source': '..\\data_files\\Seabreez Godrej Project Details.docx', 'file_type': 'docx', 'file_name': 'Seabreez Godrej Project Details.docx'}
--------------------------------------------------
Ra

In [14]:
# Generate answer using LLM
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

# Load environment variables from .env (if present)
load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("GOOGLE_API_KEY is not set. Add it to your environment or .env file.")

llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", temperature=0)

# Concatenate retrieved documents
context = "\n".join([doc['document'] for doc in retrieved_docs])

# Create prompt
prompt = f"Answer the following question based on the provided context. If the context doesn't contain enough information, say so.\n\nQuestion: {query}\n\nContext:\n{context}"

# Generate answer
response = llm.invoke(prompt)
if isinstance(response.content, list):
    answer = "\n".join(part.get("text", "") for part in response.content if isinstance(part, dict)).strip()
else:
    answer = response.content

print("Generated Answer:")
print(answer)

Generated Answer:
Based on the provided context, here are Akash Raj's skills and experience:

### **Skills**
*   **Computer Languages:** C++, Python, and Javascript.
*   **Software Packages & Tools:** NumPy, Pandas, Scikit-learn, Version-control (git), Kali Linux, TensorFlow, Django, AWS, and SQL.

### **Experience and Projects**
*   **Web Development:** Developed a web interface for document upload and metadata viewing using Django. This project involved managing eight key fields: Title, Keywords, Date, Number of Pages, Type of Document, Summary, Number of Characters, and Number of Words.
*   **Volunteer Work:** 
    *   Volunteer for the Web Development Cell, NSS IIT Roorkee.
    *   Volunteered in the National Social Summit 2024.

### **Education Background**
*   He is currently a third-year student (III Year I Semester) pursuing a **B.Tech. in Mechanical Engineering** at the **Indian Institute of Technology (IIT) Roorkee**.
